In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, 및 On-Prem(IBM Quantum Platform API를 통한) Plan 사용자만 사용할 수 있는 실험적 기능입니다. 미리보기 릴리스 상태이며 변경될 수 있습니다.

## 개요
양자 화학에서 전자 구조 문제는 전자 슈뢰딩거 방정식의 해를 찾는 것에 초점을 맞춥니다 - 시스템 전자의 거동을 설명하는 양자 파동 함수입니다. 이러한 파동 함수는 복소 진폭의 벡터이며, 각 진폭은 가능한 전자 배치의 기여에 해당합니다.

기저 상태는 시스템의 가장 낮은 에너지 파동 함수이며 분자 시스템 연구에서 특별한 중요성을 가집니다. 기저 상태를 계산하는 가장 정확한 접근 방식은 가능한 모든 전자 배치를 고려하지만, 배치의 수가 시스템 크기에 따라 기하급수적으로 증가하므로 더 큰 시스템에서는 다루기 어려워집니다.

Handover Iterative Variational Quantum Eigensolver(HI-VQE)는 분자 시스템의 기저 상태를 정확하게 추정하기 위한 혁신적인 하이브리드 양자-고전 방법입니다. 양자 하드웨어와 고전 컴퓨팅을 통합하여, 양자 프로세서를 사용해 후보 전자 배치를 효율적으로 탐색하고 고전 컴퓨터에서 결과 파동 함수를 계산합니다. 간결하면서도 화학적으로 정확한 파동 함수를 생성함으로써 HI-VQE는 양자 화학 및 재료 과학의 연구와 발견을 향상시킵니다.

![Qunova의 HI-VQE 알고리즘 개요를 보여주는 이미지](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE는 기저 상태를 높은 정확도로 효율적으로 추정하여 전자 구조 문제의 계산 복잡성을 줄입니다. 가장 관련성 높은 전자 배치의 신중하게 선택된 부분 집합에 초점을 맞추어 정확도와 효율성을 모두 최적화합니다.

고전 및 양자 컴퓨터의 장점을 결합하여 HI-VQE는 현재 추정 파동 함수를 반복적으로 개선합니다. 고유한 부분공간 구성 기법은 배치 선택을 더 효율적으로 만들어, 사용자가 양자 화학 시���레이션에서 더 큰 계산 제어력과 향상된 정확도를 가질 수 있게 합니다.

알고리즘에 대해 더 깊이 알고 싶다면 [관련 연구 논문](https://arxiv.org/abs/2503.06292)을 읽어보세요.
## 설명
분자 시스템의 전자 배치 수는 시스템 크기에 따라 기하급수적으로 증가합니다. 그러나 기저 상태와 같은 특정 전자 상태의 경우, 배치의 작은 부분만이 상태의 에너지에 크게 기여하는 것이 일반적입니다. 선택된 배치 상호작용(SCI) 방법은 이러한 희소성을 활용하여 가장 관련성 높은 배치를 식별하고 집중함으로써 계산 비용을 줄입니다. 이 배치의 부분 집합을 부분공간이라고 합니다.

HI-VQE는 분자 시스템을 표현하는 양자 컴퓨터의 고유한 효율성을 활용하여 부분공간 탐색을 지원합니다. 고전 및 양자 서브루틴을 통합하여 높은 정확도로 전자 구조 문제를 해결합니다. 기존의 양자 SCI 방법과 달리, HI-VQE는 변분 훈련, 반복적 부분공간 구성, 사전 대각화 배치 스크리닝을 결합하여 양자 측정, 반복, 고전적 대각화 비용을 줄임으로써 효율성을 향상시킵니다. 따라서 HI-VQE는 더 많은 Qubit가 필요한 더 큰 분자 시스템에 적용할 수 있으며, 동일한 정확도로 주어진 크기의 문제를 해결하는 비용을 줄입니다.

![Qunova의 HI-VQE 알고리즘의 각 단계에 대한 상세 설명을 보여주는 이미지.](../docs/images/guides/qunova-chemistry/description.avif)

시스템의 기저 상태를 계산하기 위해, HI-VQE는 먼저 고전적 화학 패키지 PySCF를 사용하여 분자 기하학 및 기타 분자 정보와 같은 사용자 제공 입력으로부터 분자 표현을 생성합니다. 그런 다음 하이브리드 양자-고전 최적화 루프에 진입하여, 포함된 배치 수를 최소화하면서 기저 상태를 최적으로 표현하도록 부분공간을 반복적으로 개선합니다. 부분공간 크기 또는 에너지 안정성과 같은 수렴 기준이 충족될 때까지 루프가 계속된 후, 계산된 기저 상태 파동 함수와 에너지가 출력됩니다. 이러한 결과는 정확한 퍼텐셜 에너지 표면을 구성하고 시스템의 추가 분석을 수행하는 데 사용할 수 있습니다.

최적화 루프는 고품질 부분공간을 생성하기 위해 양자 회로의 매개변수를 조정하는 데 초점을 맞춥니다. HI-VQE는 세 가지 양자 회로 옵션을 제공합니다: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2), [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). 최적화는 일반적인 적합성으로 인해 Hartree-Fock 참조 상태 근처에서 초기화됩니다. 그런 다음 회로가 양자 장치에서 실행되고 결과 양자 상태에서 배치가 샘플링되어 이진 문자열로 반환됩니다. 양자 장치 노이즈로 인해 일부 샘플링된 배치는 전자 수 또는 스핀을 보존하지 못하여 물리적으로 유효하지 않을 수 있습니다. HI-VQE는 [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview) 패키지의 배치 복구 프로세스를 사용하여 이를 해결하므로, 사용자는 유효하지 않은 배치를 수정하거나 폐기할 수 있습니다.

유효한 배치는 최소한으로 기여할 것으로 예측되는 배치를 제거하는 선택적 스크리닝 단계를 거칩니다. 이는 부분공간의 차원을 줄여 대각화 단계의 비용을 낮춥니다. 스크리닝이 활성화된 경우, 유효한 배치에서 예비 부분공간 해밀토니안이 구성되고 매우 느슨한 종료 기준으로 대각화가 수행됩니다. 각 배치에 대한 결과 진폭의 정확도는 낮지만, 이번 반복에서 부분공간에서 제외할 배치를 예측하는 데 효과적이며 계산이 빠릅니다.

선택된 배치가 부분공간에 추가되고, 시스템의 해밀토니안이 이 부분공간에 투영됩니다. 부분공간은 반복적으로 업데이트되며, 반복 전체에서 가장 관련성 높은 배치를 보존합니다. 이 접근 방식은 양자 회로가 각 단계에서 전체 기저 상태를 근사할 필요가 없다는 점에서 대안적 방법과 대조됩니다.

다음으로, 부분공간 해밀토니안이 고전적으로 대각화되어 가장 낮은 고유값과 해당 고유벡터를 얻으며, 이는 기저 상태와 그 에너지의 근사를 나타냅니다. 반복을 통해 부분공간 품질이 향상됨에 따라, 계산된 기저 상태는 실제 기저 상태를 더 잘 근사합니다. 이 시점에서 추가 스크리닝 단계를 수행하여 계산된 기저 상태에 상당한 기여를 하지 않는 배치를 부분공간에서 제거할 수 있습니다. 이 단계는 다음 반복으로 전달되는 부분공간이 가능한 한 간결하도록 보장합니다. 이는 대각화에서 반환된 진폭을 기반으로 평가되며, 이 진폭은 계산된 기저 상태에 대한 각 배치의 중요도 기여를 나타냅니다.

그런 다음 수렴 검사에서 추가 훈련이 결과를 개선할 수 있는지 결정합니다. 그런 경우, 선택적 고전적 확장 단계가 수행되고, 양자 회로 매개변수가 계산된 에너지를 추가로 최소화하도록 업데이트되며, 프로세스가 반복됩니다. 고전적 확장 단계는 부분공간에 대한 추가 배치를 생성하여 양자 장치에서 샘플링된 배치를 보충합니다. 먼저 대각화 결과에서 가장 큰 진폭을 가진 배치를 식별한 다음, 식별된 배치에서 단일 및 이중 여기를 사용하여 새로운 배치를 생성합니다. 그런 다음 원하는 수의 이러한 배치가 부분공간에 추가됩니다.

반복이 수렴한 것으로 결정되면, HI-VQE는 계산된 기저 상태(부분공간의 상태와 기저 상태 파동 함수에서의 진폭 형태), 에너지, 그리고 계산된 상태가 시스템 해밀토니안의 고유 상태를 형성하는지 여부를 나타내는 에너지 분산 측정값을 반환합니다.

사용자는 사용할 양자 회로와 각 양자 회로에 대해 취할 샷 수를 결정할 수 있으며, 부분공간 크기를 제어하거나 양자 생성 배치를 지원하기 위한 추가 배치의 고전적 생성을 활성화할 수 있습니다. 이러한 방식으로 사용자는 원하는 애플리케이션에 맞게 HI-VQE의 동작을 맞춤 설정할 수 있습니다.
## 시작하기
먼저 [함수 액세스를 요청](https://forms.office.com/r/zN3hvMTqJ1)하세요.
그런 다음 [IBM Quantum&reg; API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고, 이미 [계정을 저장](/guides/functions#install-qiskit-functions-catalog-client)했다고 가정하면, 다음과 같이 Qiskit Function을 선택합니다:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## 입력
함수가 허용하는 모든 입력 매개변수에 대해서는 다음 표를 참조하세요. 후속 [분자 옵션](#molecule-options) 및 [HI-VQE 옵션](#hi-vqe-options) 섹션에서 해당 인수에 대해 더 자세히 설명합니다.
| Name                   | Type                                                           | Description                                                                                                                                                                                                                                                                                                 | Required | Default                                                                  | Example                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | This can be either a string or structured lists containing atom and coordinate pairs. If this is given as a string, it must be a molecule geometry in Cartesian Coordinate Format. If this is given as a list, it should be given as a list of lists that each contain an atom string and coordinate tuple. | Yes      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` or `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Name of the backend to make the query.                                                                                                                                                                                                                                                                      | Yes      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | The maximum subspace dimension for the diagonalization. Fewer states will be used if the number is not a perfect square.                                                                                                                                                                                                                                                    | Yes      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | The maximum number of classically-generated CI states to be included in each iteration.                                                                                                                                                                                                                     | Yes      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Options related to the molecule used as input to HI-VQE. See the [Molecule options](#molecule-options) section for more details.                                                                                                                                                                                                                                                | No       | See the [Molecule options](#molecule-options) section for details.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Options controlling the behavior of the HI-VQE algorithm. See the [HI-VQE options](#hi-vqe-options) section for more details.                                                                                                                                                                                                                                                | No       | See the [HI-VQE options](#hi-vqe-options) section for details.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### 분자 옵션
다음 표는 `molecule_options` 딕셔너리에 설정할 수 있는 모든 키와 값, 데이터 유형 및 기본값을 자세히 설명합니다. 모든 키는 선택 사항입니다.

| Key                               | Value type                          | Default Value                    | Valid range                                                                                                                                                    | Explanation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Various                                                                                                                                                        | An integer specifying the total net charge of the molecular system. The default value is 0; however, it can be any integer.                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Various                                                                                                                                                        | A string specifying the basis type; these are passed to pyscf. (eg: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | Every orbital index.             | The spatial orbital indices valid for the problem.                                                                                                             | A list of active orbital indices in the interval [0, n) where n is the number of qubits used in the problem. If this is specified, then the frozen_orbitals argument must also be specified.                                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | No indices.                      | The spatial orbital indices valid for the problem, excluding active orbitals.                                                                                  | A list of frozen orbital indices in the same range as active_orbitals. If specified, then active_orbitals must also be specified. Note that only occupied orbitals should be frozen as the number of active electrons is reduced by 2 for every occupied orbital that is frozen.                                                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Hartree-Fock molecular orbitals. | Various.                                                                                                                                                       | The coefficients for the spatial orbitals used in the calculation of the electronic repulsion integrals for the system. Some valid examples are Hartree-Fock molecular orbitals, natural orbitals, and AVAS orbitals.                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` or `False`                                                                                                                                              | Used to invoke point group symmetry for the initial molecular calculations to construct the symmetry adapted orbital basis. These symmetry-adapted orbitals are used as basis functions for the following SCF calculations. The default value is False; if set to True, then it will be invoked and arbitrary point groups will automatically be detected and used. If a particular symmetry is assigned, for example, symmetry = “Dooh”, then an error will be raised if the molecular geometry is not subject to this required symmetry. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Can be used to generate a subgroup of the detected symmetry. This has no effect when symmetry is specified using the symmetry keyword argument.                                                                                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifies the measurement unit to use for atomic coordinates and distances. The default is to use angstrom units.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifies the nuclear model to be used. By default it uses the point nuclear model, and other values enable Gaussian nuclear model. If a function is given, it will be used with the Gaussian nuclear model to generate the nuclear charge distribution value 'zeta'.                                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifies the pseudopotential for the atoms in the molecule. This is None by default, indicating that no pseudopotentials are applied and all electrons are explicitly included in the calculations.                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifies whether to use cartesian GTOs as the angular momentum basis functions in the calculation. The default value of False will use spherical GTOs.                                                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | See [pyscf documentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Sets the colinear spin magnetic moment of each atom. By default, this is None and each atom is initialized with a spin of zero.                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | i.e. ["H 1s", "O 2p"] for H2O                                                                                                                                                             | This defines Atomic Orbital to be included in AVAS scheme. See [AVAS documentation](https://arxiv.org/abs/1701.07862) .                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            |  Between 0.0 and 2.0                                                                                                                                                      |  This specifies the cutoff value used to determine which Atomic Orbitals (AOs) are retained in the active space.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` or `"ccsd"`                                                                                                                                            | This defines the theoretical approach for preparing natural orbitals and selecting active orbitals based on the Natural Orbital Occupation Numbers (NOONs) scheme. See [NOONs documentation](https://doi.org/10.1063/5.0042147). Both the active and frozen orbital indices must be provided to control the number of orbitals (and the number of qubits).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
### HI-VQE 옵션
다음 표는 `hivqe_options` 딕셔너리에 설정할 수 있는 모든 키와 값, 데이터 유형 및 기본값을 자세히 설명합니다. 모든 키는 선택 사항입니다.

| Key                               | Value type                          | Default Value                    | Valid range                                                                                                                                                    | Explanation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Between 1 and 10 000.                                                                                                                                          | Number of shots to use on the quantum device per iteration.                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | Between 1 and 50.                                                                                                                                              | The maximum number of iterations to run to optimize the ansatz. The algorithm may use fewer iterations if convergence is achieved early.                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | The Hartree-Fock state.          | Binary strings with the number of bits matching the required number of qubits for the problem.                                                                 | Can be used to restart the algorithm with classical states from a previous result.                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"`, or `"lucj"`                                                                                                                                  | This specifies the quantum ansatz to optimize to generate new states. `"epa"` selects the excitation-preserving ansatz. `"hea"` selects the hardware-efficient ansatz. `"lucj"` selects the local unitary cluster Jastrow ansatz.                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | At least 2.                                                                                                                                                    | The number of iterations without significant change of the computed energy that should elapse before the algorithm is deemed to have converged.                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | More than 0 and at most 1.                                                                                                                                     | The magnitude of change in computed energy that is considered significant for the purposes of convergence checks.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` or `False`                                                                                                                                              | If `True`, the `convergence_count` iterations must occur without an interrupting significant change to qualify as converging. If `False`, then the algorithm will stop after `convergence_count` if insignificant changes have occurred at any iterations during the optimization process.                                                                                                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` or `False`.                                                                                                                                             | Whether or not to use configuration recovery from the `qiskit-addon-sqd` package. If True, invalid states sampled from the quantum device are corrected classically. If False, they are discarded.                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Any one of `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"`, or `"sca"`. If using the `"lucj"` ansatz, `"lucj_default"` is also an option. | This specifies the entanglement scheme that should be used within the quantum circuit, following common Qiskit and [ffsim conventions for the LUCJ ansatz](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Greater than 0.                                                                                                                                                | The number of repetitions of each layer in the quantum circuit.                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | At least 0, and less than 1.                                                                                                                                   | The tolerance for deciding which states should be screened out of the subspace after diagonalization. It specifies the inclusion threshold for the subspace states based on their computed amplitudes.                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Between `1e-4` and `1e-1`, inclusive.                                                                                                                          | The tolerance for predicting which states should be screened out of the subspace prior to diagonalization. It controls the accuracy of the predicted amplitudes for each state, with a smaller value resulting in more accurate predictions.                                                                                                                                                                                                                                                                                            |
## 출력
함수는 네 개의 키와 값을 가진 딕셔너리를 반환합니다. 키와 값은 다음 표에 문서화되어 있습니다:

| Key             | Value Type    | Explanation                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | The approximate ground state energy of the molecule.                                                                      |
| `"states"`      | `List[str]`   | The selected determinants that form the subspace used to solve for the energy. They are in alternating alpha-beta format. |
| `"eigenvector"` | `List[float]` | The eigenvector corresponding to the ground state of the subspace composed of `"states"`.                                 |
| `"energy_variance"` | `float` | The energy variance of the ground state of the subspace composed of `"states"`, giving an indication of the quality of the solution. This value is non-negative and a lower value means that the ground state of the subspace more closely approximates an eigenstate of the system's Hamiltonian. |
| `"energy_history"` | `List[float]` | The energies computed each iteration during the hybrid optimization process, in the same order that they were computed. Two energies are computed per iteration as part of the SPSA optimization process. |
## 예제
첫 번째 예제는 HI-VQE 알고리즘을 사용하여 NH3 분자의 기저 상태 에너지를 계산하는 방법을 보여줍니다.
#### 분자 기하학 및 옵션 정의
NH3의 분자 기하학은 각 원자에 대해 ";"로 구분된 데카르트 좌표로 제공됩니다.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

분자 시스템에 대한 추가 옵션은 다음 딕셔너리 형식으로 정의하고 제공할 수 있습니다.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

기하학 및 옵션 입력으로 함수를 실행합니다.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

문제가 발생했을 때 지원 요청에 제공할 수 있도록 Function 작업 ID를 출력하는 것이 좋습니다.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


이 예제는 NH3 분자에 대해 sto3g 기저의 8개 오비탈과 16개 Qubit를 활용합니다.
Qiskit Function 워크로드의 [상태](/guides/functions#check-job-status)를 확인하거나 다음과 같이 [결과](/guides/functions#retrieve-results)를 반환합니다:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


작업이 완료된 후 `result()` 인스턴스로 결과를 얻을 수 있습니다.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

기저 상태 에너지에 접근하려면 "energy" 키를 사용하세요. "eigenvector" 키는 결과의 "states"에 저장된 전자 배치의 해당 비트스트링 표기법과 함께 CI 계수를 제공합니다.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

출력:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## 성능
이 섹션에서는 Li2S에 대한 24-Qubit 케이스, N2 분자에 대한 40-Qubit 케이스, FeP-NO 시스템에 대한 44-Qubit 케이스로 시연된 HI-VQE의 벤치마크 계산을 보여줍니다.
#### 24 Qubit Li2S 분자의 해리 퍼텐셜 에너지 표면 곡선
PES 곡선은 FCI 참조 및 RHF의 초기 추정과 함께 FCI 참조로부터의 에너지 오차와 함께 표시됩니다.

![HI-VQE가 Li2S 시스템에 대한 고전적 참조 PES 곡선의 화학적 정확도 내에서 솔루션을 생성함을 보여주는 이미지.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

계산은 다음 기하학 및 옵션으로 수행되었습니다.